# PIMCO AI Surveillance Analysis Based on ABS Prospectuses

### Milestones
- **Milestone 1:** Data collection, parsing, metadata filtering, and keyword/regex search
- **Milestone 2:** Embedding-based semantic search and Q&A
- **Milestone 3:** User interface and evaluation

In [21]:
import os
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup 

In [22]:
from pathlib import Path

# Should get the project root
PROJECT_ROOT = Path.cwd().parent
print("Project root:", PROJECT_ROOT)

Project root: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses


In [23]:
raw_data = PROJECT_ROOT / "data" / "raw"
processed_data = PROJECT_ROOT / "data" / "processed"
manifest_path = PROJECT_ROOT / "data" / "manifest.csv"

# Output file
output = processed_data / "abs_index.csv"

print("Raw data:", raw_data)
print("Manifest:", manifest_path)
print("Processed data:", processed_data)
print("Output file:", output)

Raw data: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/raw
Manifest: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/manifest.csv
Processed data: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed
Output file: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/abs_index.csv


## Milestone 1 — Data Collection and Text Extraction

The dataset contains SEC ABS prospectus filings in HTML (`.htm`) format.

For each filing, we:
1. Locate the HTML file
2. Read the HTML
3. Remove HTML formatting
4. Extract readable text
5. Store the result in a DataFrame

In [24]:
def get_htm_files(data_folder):
    data_folder = Path(data_folder)
    
    if not data_folder.exists():
        raise FileNotFoundError(
            f"Data folder not found: {data_folder}"
        )
    
    htm_files = sorted(
        file
        for file in data_folder.iterdir()
        if file.is_file() and file.suffix.lower() == ".htm"
    )
    
    return htm_files


htm_files = get_htm_files(raw_data)

print(f"Found {len(htm_files)} HTM filings.")

Found 122 HTM filings.


In [25]:
# Extract readable text from an SEC HTML filing.    
def extract_text_from_htm(file_path):
   with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:
        
        html = file.read()
        soup = bsoup(
        html,
        "html.parser"
    )
        text = soup.get_text(
        " ",
        strip=True
    )
        return text

In [6]:
data = []

for file_path in htm_files:
    
    text = extract_text_from_htm(
        file_path
    )
    
    data.append({
        "file_name": file_path.name,
        "extracted_text": text
    })

df = pd.DataFrame(data)

print("Number of filings:", len(df))
print("Number of columns:", len(df.columns))

df.head()

Number of filings: 122
Number of columns: 2


,file_name,extracted_text
0,AFS_SENSUB_CORP__2024-05-15_424H.htm,424H 1 d679938d424h.htm 424H 424H Table of Con...
1,AFS_SENSUB_CORP__2024-06-26_424H.htm,424H 1 d792078d424h.htm 424H 424H Table of Con...
2,AFS_SENSUB_CORP__2024-10-02_424H.htm,424H 1 d847362d424h.htm 424H 424H Table of Con...
3,AFS_SENSUB_CORP__2025-01-06_424H.htm,424H 1 d909372d424h.htm 424H 424H Table of Con...
4,AFS_SENSUB_CORP__2025-05-01_424H.htm,424H 1 d111258d424h.htm 424H 424H Table of Con...


In [27]:
# metadata
def filename_metadata(df):
    df = df.copy()
    # Filing date
    df["filing_date"] = df["file_name"].str.extract(
        r"(\d{4}-\d{2}-\d{2})"
    )
    # Filing type
    df["filing_type"] = df["file_name"].str.extract(
        r"_(424H)\.htm$"
    )
    # Deal name
    df["deal_name"] = (
        df["file_name"]
        .str.replace(
            r"_\d{4}-\d{2}-\d{2}_424H\.htm$",
            "",
            regex=True
        )
        .str.replace(
            "_",
            " ",
            regex=False
        )
    )
    return df
df = filename_metadata(df)
df[
    [
        "file_name",
        "deal_name",
        "filing_date",
        "filing_type"
    ]
].tail()

,file_name,deal_name,filing_date,filing_type
117,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2025-10-01_424...,WORLD OMNI AUTO RECEIVABLES LLC,2025-10-01,424H
118,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-02-02_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-02-02,424H
119,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-04-29_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-04-29,424H
120,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-07-13_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-07-13,424H
121,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-08-12_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-08-12,424H


In [28]:
# Read manifest csv
if not manifest_path.exists():
    raise FileNotFoundError(
        f"Manifest not found: {manifest_path}"
    )
manifest = pd.read_csv(
    manifest_path
)
print("Manifest rows:", len(manifest))
print("Manifest columns:")
print(manifest.columns.tolist())

# Get actual filename from local_path
manifest["file_name"] = (
    manifest["local_path"]
    .astype(str)
    .str.split(r"[\\/]")
    .str[-1]
)

Manifest rows: 129
Manifest columns:
['entity_name', 'cik', 'form', 'filing_date', 'accession', 'accession_clean', 'primary_doc', 'description', 'doc_url', 'index_url', 'local_path', 'downloaded']


In [29]:
manifest["file_name"] = (
    manifest["local_path"]
    .astype(str)
    .str.split(r"[\\/]")
    .str[-1]
)

In [31]:
# Remove old SEC metadata columns from df
metadata_columns = [
    "cik",
    "accession",
    "primary_doc",
    "doc_url",
    "cik_x",
    "cik_y",
    "accession_x",
    "accession_y",
    "primary_doc_x",
    "primary_doc_y",
    "doc_url_x",
    "doc_url_y"
]
df = df.drop(
    columns=[
        col for col in metadata_columns
        if col in df.columns
    ],
    errors="ignore"
)

# Keep one manifest row per filename
manifest_unique = (
    manifest
    .drop_duplicates("file_name")
)
print(
    "Unique manifest filings:",
    len(manifest_unique)
)

# Merge SEC metadata into df
df = df.merge(
    manifest_unique[
        [
            "file_name",
            "cik",
            "accession",
            "primary_doc",
            "doc_url"
        ]
    ],
    on="file_name",
    how="left"
    )
print(
        "SEC metadata matched:",
        df["cik"].notna().sum(),
        "/",
        len(df)
    )

Unique manifest filings: 122
SEC metadata matched: 122 / 122


In [32]:
# Normalize whitespace in extracted filing text
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    
    text = re.sub(
        r"\s+",
        " ",
        text
    )
    return text.strip()

In [33]:
df["clean_text"] = (
    df["extracted_text"]
    .apply(clean_text)
)

In [34]:
# Character count
df["text_length"] = (
    df["clean_text"]
    .str.len()
)
# Word count
df["word_count"] = (
    df["clean_text"]
    .str.split()
    .str.len()
)
# Convert filing date to datetime
df["filing_date"] = pd.to_datetime(
    df["filing_date"],
    errors="coerce"
)
# Extract filing year
df["filing_year"] = (
    df["filing_date"]
    .dt.year
)

In [35]:

print("DATA QUALITY CHECK: ")
print(
    "Total filings:",
    len(df)
)
print(
    "Empty extracted text:",
    (df["text_length"] == 0).sum()
)
print(
    "Missing filing dates:",
    df["filing_date"].isna().sum()
)
print(
    "Missing deal names:",
    df["deal_name"].isna().sum()
)
print(
    "Missing CIK:",
    df["cik"].isna().sum()
)
print(
    "Missing SEC URLs:",
    df["doc_url"].isna().sum()
)

print(
    "Average words per filing:",
    round(df["word_count"].mean())
)
print(
    "Shortest filing:",
    df["word_count"].min(),
    "words"
)
print(
    "Longest filing:",
    df["word_count"].max(),
    "words"
)

DATA QUALITY CHECK: 
Total filings: 122
Empty extracted text: 0
Missing filing dates: 0
Missing deal names: 0
Missing CIK: 0
Missing SEC URLs: 0
Average words per filing: 147154
Shortest filing: 83055 words
Longest filing: 586179 words


In [36]:
df.to_csv(
    output,
    index=False
)
print(
    f"Saved processed index to:\n{output}"
)
print(
    f"Number of filings: {len(df)}"
)
print(
    f"Number of columns: {len(df.columns)}"
)

Saved processed index to:
/Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/abs_index.csv
Number of filings: 122
Number of columns: 13


In [37]:
def extract_snippets(
    text,
    pattern,
    snippets_per_result=2,
    context_chars=250
):
    """
    Extract text surrounding each search match.
    """
    
    if not isinstance(text, str):
        return []
    
    try:
        compiled_pattern = re.compile(
            pattern,
            flags=re.IGNORECASE
        )
    except re.error:
        return []
    
    snippets = []
    
    for match in compiled_pattern.finditer(text):
        
        start = max(
            0,
            match.start() - context_chars
        )
        
        end = min(
            len(text),
            match.end() + context_chars
        )
        
        snippet = text[start:end].strip()
        
        if start > 0:
            snippet = "..." + snippet
        
        if end < len(text):
            snippet += "..."
        
        snippets.append(snippet)
        
        if len(snippets) >= snippets_per_result:
            break
    
    return snippets

In [38]:
def search_filings(
    query,
    year=None,
    filing_type=None,
    regex=False,
    max_results=10,
    snippets_per_result=2
):
    """
    Search ABS prospectuses using keywords,
    phrases, or regular expressions.
    """
    
    # Start with the processed dataset
    search_df = df.copy()
    
    # ------------------------------
    # Metadata filtering
    # ------------------------------
    
    if year is not None:
        
        search_df = search_df[
            search_df["filing_year"] == int(year)
        ]
    
    if filing_type is not None:
        
        search_df = search_df[
            search_df["filing_type"]
            .fillna("")
            .str.lower()
            ==
            str(filing_type).lower()
        ]
    
    # ------------------------------
    # Create search pattern
    # ------------------------------
    
    if regex:
        pattern = query
    
    else:
        pattern = re.escape(query)
    
    try:
        compiled_pattern = re.compile(
            pattern,
            flags=re.IGNORECASE
        )
    
    except re.error as error:
        raise ValueError(
            f"Invalid regular expression: {error}"
        )
    
    # ------------------------------
    # Count matches
    # ------------------------------
    
    search_df["matches"] = (
        search_df["clean_text"]
        .fillna("")
        .apply(
            lambda text:
            len(
                compiled_pattern.findall(text)
            )
        )
    )
    
    # Keep only matching filings
    search_df = search_df[
        search_df["matches"] > 0
    ].copy()
    
    # ------------------------------
    # Supporting snippets
    # ------------------------------
    
    search_df["supporting_snippets"] = (
        search_df["clean_text"]
        .fillna("")
        .apply(
            lambda text:
            extract_snippets(
                text,
                pattern,
                snippets_per_result
            )
        )
    )
    
    # ------------------------------
    # Rank results
    # ------------------------------
    
    search_df = search_df.sort_values(
        by="matches",
        ascending=False
    )
    
    # ------------------------------
    # Return useful columns
    # ------------------------------
    
    columns = [
        "deal_name",
        "filing_date",
        "filing_year",
        "filing_type",
        "matches",
        "supporting_snippets",
        "doc_url",
        "file_name"
    ]
    
    return (
        search_df[columns]
        .head(max_results)
        .reset_index(drop=True)
    )

In [39]:
results = search_filings(
    "custody",
    max_results=10
)
results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-03-23,2026,424H,15,[...assu Whole Loans 240 Certain Rights of eac...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-15,2026,424H,13,[...ssu Whole Loans 246 Certain Rights of each...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
6,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-05,2026,424H,10,[...Passu Whole Loans 243 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
7,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
8,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-02-11,2026,424H,9,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
9,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-06-10,2026,424H,8,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...


In [40]:
results = search_filings(
    "physical custody",
    max_results=10
)

results


,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name


In [41]:
results = search_filings(
    r"\b(physical|paper)\b.{0,100}\b(custody|custodian)\b",
    regex=True,
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,SANTANDER DRIVE AUTO RECEIVABLES LLC,2026-02-11,2026,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2026-02-1...
1,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-11-06,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-11-0...
2,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-09-10,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-09-1...
3,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-07-17,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-07-1...
4,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-05-14,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-05-1...
5,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-03-12,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-03-1...
6,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-01-10,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-01-1...
7,Bridgecrest Auto Funding LLC,2026-07-15,2026,424H,1,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/197482...,Bridgecrest_Auto_Funding_LLC_2026-07-15_424H.htm
8,SANTANDER DRIVE AUTO RECEIVABLES LLC,2024-10-10,2024,424H,1,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2024-10-1...
9,SANTANDER DRIVE AUTO RECEIVABLES LLC,2024-09-11,2024,424H,1,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2024-09-1...


In [42]:
results = search_filings(
    "custody",
    year=2025,
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,BMW AUTO LEASING LLC,2025-05-29,2025,424H,6,[...s 128 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-05-29_424H.htm
6,BMW AUTO LEASING LLC,2025-10-02,2025,424H,6,[...s 129 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-10-02_424H.htm
7,HARLEY-DAVIDSON CUSTOMER FUNDING CORP,2025-05-14,2025,424H,3,[...e Uniform Commercial Code. As described un...,https://www.sec.gov/Archives/edgar/data/111492...,HARLEY-DAVIDSON_CUSTOMER_FUNDING_CORP__2025-05...
8,CARMAX AUTO FUNDING LLC,2025-09-09,2025,424H,2,[...r of this securitization transaction and w...,https://www.sec.gov/Archives/edgar/data/125938...,CARMAX_AUTO_FUNDING_LLC_2025-09-09_424H.htm
9,TOYOTA AUTO FINANCE RECEIVABLES LLC,2025-10-08,2025,424H,2,[...behalf of their participants through their...,https://www.sec.gov/Archives/edgar/data/113113...,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2025-10-08...


In [43]:
results = search_filings(
    "custody",
    filing_type="424H",
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-03-23,2026,424H,15,[...assu Whole Loans 240 Certain Rights of eac...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-15,2026,424H,13,[...ssu Whole Loans 246 Certain Rights of each...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
6,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-05,2026,424H,10,[...Passu Whole Loans 243 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
7,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
8,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-02-11,2026,424H,9,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
9,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-06-10,2026,424H,8,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...


In [44]:
results = search_filings(
    "custody",
    year=2025,
    filing_type="424H",
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,BMW AUTO LEASING LLC,2025-05-29,2025,424H,6,[...s 128 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-05-29_424H.htm
6,BMW AUTO LEASING LLC,2025-10-02,2025,424H,6,[...s 129 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-10-02_424H.htm
7,HARLEY-DAVIDSON CUSTOMER FUNDING CORP,2025-05-14,2025,424H,3,[...e Uniform Commercial Code. As described un...,https://www.sec.gov/Archives/edgar/data/111492...,HARLEY-DAVIDSON_CUSTOMER_FUNDING_CORP__2025-05...
8,CARMAX AUTO FUNDING LLC,2025-09-09,2025,424H,2,[...r of this securitization transaction and w...,https://www.sec.gov/Archives/edgar/data/125938...,CARMAX_AUTO_FUNDING_LLC_2025-09-09_424H.htm
9,TOYOTA AUTO FINANCE RECEIVABLES LLC,2025-10-08,2025,424H,2,[...behalf of their participants through their...,https://www.sec.gov/Archives/edgar/data/113113...,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2025-10-08...


In [45]:
test_queries = [
    "physical",
    "paper",
    "custody",
    "custodian"
]

for query in test_queries:
    
    results = search_filings(
        query,
        max_results=5
    )
    
    print("=" * 60)
    print(f"QUERY: {query}")
    print(f"MATCHING FILINGS: {len(results)}")
    
    if len(results) > 0:
        
        top_result = results.iloc[0]
        
        print(
            "TOP DEAL:",
            top_result["deal_name"]
        )
        
        print(
            "DATE:",
            top_result["filing_date"]
        )
        
        print(
            "MATCHES:",
            top_result["matches"]
        )
        
        print("\nSNIPPET:")
        
        print(
            top_result["supporting_snippets"][0]
        )
        
    else:
        
        print("No results found.")

QUERY: physical
MATCHING FILINGS: 5
TOP DEAL: WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC
DATE: 2026-02-11 00:00:00
MATCHES: 59

SNIPPET:
...ptible to damage or wear and tear than other types of multifamily housing, the reliance on the financial well-being of the college or university to which it relates, competition from on campus housing units, which may adversely affect occupancy, the physical layout of the housing, which may not be readily convertible to traditional multifamily use, and that student tenants have a higher turnover rate than other types of multifamily tenants, which in certain cases is compounded by the fact that student l...
QUERY: paper
MATCHING FILINGS: 5
TOP DEAL: BMW AUTO LEASING LLC
DATE: 2025-05-29 00:00:00
MATCHES: 19

SNIPPET:
...eness of the settlement pending PIMCO’s appeal of its objection. The case has been referred to the circuit mediator, and, on February 24, 2025, the Consumer Financial Protection Bureau sought extension of time for 90 days to prep

In [47]:
df.head()
df.columns.tolist()

['file_name',
 'extracted_text',
 'filing_date',
 'filing_type',
 'deal_name',
 'cik',
 'accession',
 'primary_doc',
 'doc_url',
 'clean_text',
 'text_length',
 'word_count',
 'filing_year']